# GutBrainIE – NER Ensemble (BERT token-classifier + GLiNER v2 span)
Obiettivo: **massimizzare i risultati finali** senza rifare training.

- BERT = *precision anchor*
- GLiNER v2 = *recall booster*

Pipeline:
1. Carica DEV
2. Predizioni BERT (dal tuo modello già allenato)
3. Predizioni GLiNER v2 (adapter LoRA già allenato)
4. Ensemble a livello di span + pruning finale
5. Valutazione con **`evaluate.py` ufficiale**


In [1]:
# 0) Imports
import os
import json
from pathlib import Path
from typing import Any, Dict, List
from collections import defaultdict
import re
import torch
import random
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification

C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Config

In [2]:
# =============================
# 1) CONFIG (EDIT ME)
# =============================
DATA_DIR = Path(r"C:/Users/super/Documents/UniPd/ATA/GutBrainIE/data/GutBrainIE_Full_Collection_2025/Annotations")
DEV_JSON = DATA_DIR / "Dev/json_format/dev.json"

# ---- BERT fine-tuned
BERT_MODEL_DIR = Path("models/bert_biomedbert_ner_label_weight")

# ---- GLiNER v2 fine-tuned (LoRA adapter directory)
GLINER_BASE_MODEL = "fastino/gliner2-base-v1"
GLINER_ADAPTER_DIR = Path(r"C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/ner/models/gliner_v2_finetuned/best")

# Output predictions
OUT_PRED_DIR = Path(r"C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions")
OUT_PRED_DIR.mkdir(parents=True, exist_ok=True)

# ---- GLiNER inference threshold (tune for ensemble)
GLINER_THRESHOLD = 0.20          # 0.20 => più recall, 0.33 => più preciso
GLINER_INCLUDE_CONFIDENCE = True

# ---- Ensemble knobs
GLINER_MIN_ADD_SCORE = 0.20      # filtra aggiunte GLiNER troppo rumorose
ALLOW_EXPAND_SAME_LABEL = True   # se GLiNER contiene BERT (stessa label), espandi
EXPAND_MIN_SCORE = 0.60

APPLY_FINAL_PRUNE = True         # risolve overlap finali (BERT > GLiNER)

# evaluate.py (ufficiale). Metti qui il path nel tuo repo, se diverso.
EVALUATE_PY = Path("src/evaluate.py")
INCLUDE_CONFIDENCE = True


## 2) Labels (come nei tuoi notebook)

In [3]:
ENTITY_LABELS: List[str] = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "disease, disorder or finding",  # -> DDF
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
]

LABEL_MAPPING = {"disease, disorder or finding": "DDF"}

def normalize_label(label: str) -> str:
    return LABEL_MAPPING.get(label, label)

LEGAL_ENTITY_LABELS = {
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
}

## Load DEV

In [4]:
def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

dev_data = load_json(DEV_JSON)
print("Loaded DEV docs:", len(dev_data))

Loaded DEV docs: 40


## BERT: Load model + tokenizer (inference only)

In [5]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_DIR)
bert_model = AutoModelForTokenClassification.from_pretrained(BERT_MODEL_DIR).to(device)
bert_model.eval()

id2label = bert_model.config.id2label
label2id = bert_model.config.label2id
print("num labels:", len(id2label))


device: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 459.61it/s, Materializing param=classifier.weight]                                      


num labels: 27


### BERT: Two-pass thresholds (precision first, recall second)

In [6]:
LABEL_THRESH_HIGH = {
    "DDF": 0.88,
    "bacteria": 0.88,
    "statistical technique": 0.92,
    "biomedical technique": 0.82,
    "gene": 0.75,
    "food": 0.70,
    "chemical": 0.80,
    "dietary supplement": 0.85,
    "drug": 0.80,
    "microbiome": 0.78,
    "anatomical location": 0.78,
    "human": 0.70,
    "animal": 0.70,
}

LABEL_THRESH_RECALL = dict(LABEL_THRESH_HIGH)
LABEL_THRESH_RECALL.update({
    "DDF": 0.84,
    "chemical": 0.74,
    "biomedical technique": 0.76,
    "gene": 0.70,
    "food": 0.62,
})

DEFAULT_THRESH = 0.80

RECALL_LABELS = {"chemical", "biomedical technique", "gene", "food"}  # aggiungi "DDF" se serve

## BERT: Simple filters + label-specific postprocess

In [7]:
BAD_BACTERIA = {"bacteria", "micro", "microbes", "microorganisms", "genera", "taxa"}
BAD_CHEMICAL = {"metabolites", "neurotransmitters"}
BAD_DIETSUPP = {"nnss"}

def normalize_span(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def apply_simple_filters(entities):
    cleaned = []
    for e in entities:
        s = normalize_span(e.get("text_span", ""))

        # drop HTML/markup garbage
        if "<" in s or ">" in s:
            continue

        # drop empty/very short spans
        if len(s) <= 1:
            continue

        # bacteria generic junk
        if e["label"] == "bacteria" and s in BAD_BACTERIA:
            continue

        # dietary supplement junk
        if e["label"] == "dietary supplement" and s in BAD_DIETSUPP:
            continue

        # chemical generic junk
        if e["label"] == "chemical" and s in BAD_CHEMICAL:
            continue

        cleaned.append(e)
    return cleaned


GENE_LIKE = re.compile(
    r"^(il-\d+|tnf-?α|ifn-?γ|tgf-?β\d*|snca|park7|dj-1|α-?synuclein|p-?α-?synuclein)$",
    re.IGNORECASE,
)

def postprocess_gene_vs_chemical(entities):
    for e in entities:
        if e["label"] == "chemical":
            s = normalize_span(e.get("text_span", ""))
            if GENE_LIKE.match(s):
                e["label"] = "gene"
    return entities

### BERT core: decode BIO + confidence score

In [8]:
## BERT: Core predictor (BIO decode + confidence score)
def predict_entities_with_scores(
        model,
        tokenizer,
        text: str,
        id2label: dict,
        label2id: dict,
        max_length: int = 512,
):
    """
    Output entity schema:
      start_idx (inclusive), end_idx (inclusive), label, text_span, score

    Score = mean token probability over the entity span.

    BIO repair:
      - I-X without active entity => start new entity as X
      - I-X with different active label => close current and start X
    """
    if not text:
        return []

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=max_length,
    )

    offsets = enc.pop("offset_mapping")[0].cpu().numpy()
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)
        logits = out.logits[0]  # [T, C]
        probs = torch.softmax(logits, dim=-1)  # [T, C]
        pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()
        probs_cpu = probs.cpu().numpy()

    labels = [id2label[int(i)] for i in pred_ids]

    entities = []
    current = None

    def _start_entity(ent_label: str, s: int, e: int, t_idx: int):
        prob_idx = label2id.get(f"B-{ent_label}", None)
        token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())
        return {
            "start_idx": int(s),
            "end_idx": int(e) - 1,  # inclusive
            "label": ent_label,
            "text_span": text[int(s):int(e)],
            "_token_probs": [token_prob],
        }

    def _extend_entity(ent: dict, e: int, t_idx: int):
        ent_label = ent["label"]
        prob_idx = label2id.get(f"I-{ent_label}", None)
        token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())
        ent["end_idx"] = int(e) - 1
        ent["text_span"] = text[ent["start_idx"]:int(e)]
        ent["_token_probs"].append(token_prob)

    for t_idx, (lab, (s, e)) in enumerate(zip(labels, offsets)):
        s = int(s);
        e = int(e)

        # special tokens
        if s == 0 and e == 0:
            continue
        if e <= s:
            continue

        if lab.startswith("B-"):
            if current is not None:
                entities.append(current)
            ent_label = lab[2:]
            current = _start_entity(ent_label, s, e, t_idx)

        elif lab.startswith("I-"):
            ent_label = lab[2:]

            if current is None:
                current = _start_entity(ent_label, s, e, t_idx)
                continue

            if ent_label != current["label"]:
                entities.append(current)
                current = _start_entity(ent_label, s, e, t_idx)
                continue

            _extend_entity(current, e, t_idx)

        else:
            if current is not None:
                entities.append(current)
                current = None

    if current is not None:
        entities.append(current)

    # add score
    for ent in entities:
        probs_list = ent.pop("_token_probs", [])
        ent["score"] = float(np.mean(probs_list)) if probs_list else 0.0

    return entities


### BERT: threshold filter + merge policy (two-pass)

In [9]:
## BERT: Threshold filter + merge policy
def filter_by_threshold_with_map(entities, label_thresh):
    out = []
    for e in entities:
        thr = label_thresh.get(e["label"], DEFAULT_THRESH)
        if float(e.get("score", 0.0)) >= float(thr):
            out.append(e)
    return out


def overlaps(a, b):
    # inclusive spans
    return not (a["end_idx"] < b["start_idx"] or b["end_idx"] < a["start_idx"])


def any_overlap(ent, kept):
    for k in kept:
        if k["location"] != ent["location"]:
            continue
        if overlaps(ent, k):
            return True
    return False


def merge_two_pass(ents_high, ents_rec, recall_labels):
    kept = list(ents_high)
    for e in ents_rec:
        if e["label"] not in recall_labels:
            continue
        if not any_overlap(e, kept):
            kept.append(e)
    return kept


#### BERT: Predict entities for a single segment (title/abstract) with two-pass

In [10]:
def predict_segment_entities_two_pass(model, tokenizer, text, location):
    ents_raw = predict_entities_with_scores(
        model=model,
        tokenizer=tokenizer,
        text=text,
        id2label=id2label,
        label2id=label2id,
        max_length=512,
    )

    # pass 1 (high precision)
    ents_high = filter_by_threshold_with_map(ents_raw, LABEL_THRESH_HIGH)
    ents_high = apply_simple_filters(ents_high)
    ents_high = postprocess_gene_vs_chemical(ents_high)
    for e in ents_high:
        e["location"] = location

    # pass 2 (recall)
    ents_rec = filter_by_threshold_with_map(ents_raw, LABEL_THRESH_RECALL)
    ents_rec = apply_simple_filters(ents_rec)
    ents_rec = postprocess_gene_vs_chemical(ents_rec)
    for e in ents_rec:
        e["location"] = location

    merged = merge_two_pass(ents_high, ents_rec, recall_labels=RECALL_LABELS)

    # ⚠️ NON togliere lo score qui (serve per l'ensemble).
    # Lo toglierai solo quando salvi la submission finale.
    return merged

### BERT: Predict DEV dataset (pmid -> {entities})

In [11]:
from tqdm import tqdm

def predict_dataset_bert_two_pass(dataset: Dict[str, Any]) -> Dict[str, Any]:
    preds = {}

    for pmid, article in tqdm(dataset.items(), desc="BERT two-pass inference"):
        title = (article.get("metadata", {}) or {}).get("title", "") or ""
        abstract = (article.get("metadata", {}) or {}).get("abstract", "") or ""

        ents = []
        if title.strip():
            ents += predict_segment_entities_two_pass(bert_model, bert_tokenizer, title, "title")
        if abstract.strip():
            ents += predict_segment_entities_two_pass(bert_model, bert_tokenizer, abstract, "abstract")

        # dedupe
        seen = set()
        dedup = []
        for e in ents:
            # dedupe key ignores text_span/score (score cambia spesso)
            k = (int(e["start_idx"]), int(e["end_idx"]), str(e["location"]), str(e["label"]))
            if k in seen:
                continue
            seen.add(k)
            dedup.append(e)

        preds[pmid] = {"entities": dedup}

    return preds

bert_predictions = predict_dataset_bert_two_pass(dev_data)
print("BERT docs predicted:", len(bert_predictions))
print("BERT total entities:", sum(len(v["entities"]) for v in bert_predictions.values()))

BERT two-pass inference: 100%|██████████| 40/40 [00:01<00:00, 21.75it/s]

BERT docs predicted: 40
BERT total entities: 980


##  GLiNER v2 inference

In [12]:
## GLiNER v2 inference (load base + LoRA adapter, predict dev)
from gliner2 import GLiNER2
from typing import Tuple


def dedupe_entities(ents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for e in ents:
        k = (int(e["start_idx"]), int(e["end_idx"]), str(e["location"]), str(e["label"]))
        if k in seen:
            continue
        seen.add(k)
        out.append(e)
    return out


def merge_adjacent_same_label(ents: List[Dict[str, Any]], title: str, abstract: str) -> List[Dict[str, Any]]:
    by_loc: Dict[str, List[Dict[str, Any]]] = {}
    for e in ents:
        by_loc.setdefault(e["location"], []).append(e)

    merged_all: List[Dict[str, Any]] = []
    for loc, group in by_loc.items():
        text = title if loc == "title" else abstract
        group = sorted(group, key=lambda x: (x["label"], x["start_idx"], x["end_idx"]))

        i = 0
        while i < len(group):
            cur = dict(group[i])
            i += 1

            while i < len(group) and group[i]["label"] == cur["label"]:
                nxt = group[i]

                # contiguous
                if nxt["start_idx"] == cur["end_idx"] + 1:
                    cur["end_idx"] = nxt["end_idx"]
                    cur["score"] = max(float(cur.get("score", 0.0)), float(nxt.get("score", 0.0)))
                    i += 1
                    continue

                # separated by one space
                if (
                        nxt["start_idx"] == cur["end_idx"] + 2
                        and text
                        and text[cur["end_idx"] + 1: cur["end_idx"] + 2] == " "
                ):
                    cur["end_idx"] = nxt["end_idx"]
                    cur["score"] = max(float(cur.get("score", 0.0)), float(nxt.get("score", 0.0)))
                    i += 1
                    continue

                break

            # recompute span from text if possible
            if text and 0 <= int(cur["start_idx"]) <= int(cur["end_idx"]) < len(text):
                cur["text_span"] = text[int(cur["start_idx"]): int(cur["end_idx"]) + 1]

            merged_all.append(cur)

    return merged_all


def postprocess_entities(ents: List[Dict[str, Any]], title: str, abstract: str) -> List[Dict[str, Any]]:
    # NB: volutamente NON facciamo prune overlap (ti dava soft recall migliore senza)
    ents = dedupe_entities(ents)
    ents = merge_adjacent_same_label(ents, title=title, abstract=abstract)
    ents = dedupe_entities(ents)
    return ents


def gliner2_extract_spans(
        extractor: GLiNER2,
        text: str,
        labels: List[str],
        threshold: float,
        location: str,
        include_confidence: bool = True,
) -> List[Dict[str, Any]]:
    if not text:
        return []

    result = extractor.extract_entities(
        text,
        labels,
        threshold=threshold,
        include_spans=True,
        include_confidence=include_confidence,
    )

    formatted: List[Dict[str, Any]] = []
    for raw_label, items in (result.get("entities", {}) or {}).items():
        norm_label = normalize_label(raw_label)
        if norm_label not in LEGAL_ENTITY_LABELS:
            continue

        for it in items:
            start = int(it["start"])
            end_exclusive = int(it["end"])
            end_inclusive = end_exclusive - 1  # GutBrainIE uses inclusive

            formatted.append(
                {
                    "start_idx": start,
                    "end_idx": end_inclusive,
                    "location": location,
                    "text_span": it.get("text", text[start:end_exclusive]),
                    "label": norm_label,
                    "score": float(it.get("confidence", 1.0)),
                }
            )

    return formatted


def predict_dataset_gliner2(
        extractor: GLiNER2,
        dataset: Dict[str, Any],
        labels: List[str],
        threshold: float,
        include_confidence: bool = True,
) -> Dict[str, Any]:
    preds: Dict[str, Any] = {}

    for pmid, article in tqdm(dataset.items(), desc=f"GLiNER2 inference (th={threshold})"):
        title = (article.get("metadata", {}) or {}).get("title", "") or ""
        abstract = (article.get("metadata", {}) or {}).get("abstract", "") or ""

        ents: List[Dict[str, Any]] = []
        ents += gliner2_extract_spans(extractor, title, labels, threshold, "title",
                                      include_confidence=include_confidence)
        ents += gliner2_extract_spans(extractor, abstract, labels, threshold, "abstract",
                                      include_confidence=include_confidence)

        ents = postprocess_entities(ents, title=title, abstract=abstract)

        preds[pmid] = {
            "entities": [
                {k: e[k] for k in ["start_idx", "end_idx", "location", "text_span", "label", "score"]}
                for e in ents
            ]
        }

    return preds


## Load GLiNER base + adapter
gliner_model = GLiNER2.from_pretrained(GLINER_BASE_MODEL)

if hasattr(gliner_model, "load_adapter"):
    gliner_model.load_adapter(str(GLINER_ADAPTER_DIR))
elif hasattr(gliner_model, "load_lora_adapter"):
    gliner_model.load_lora_adapter(str(GLINER_ADAPTER_DIR))
else:
    raise RuntimeError("No adapter loading method found (load_adapter/load_lora_adapter).")

print("Loaded GLiNER adapter:", GLINER_ADAPTER_DIR)

## Run GLiNER on DEV
gliner_predictions = predict_dataset_gliner2(
    extractor=gliner_model,
    dataset=dev_data,
    labels=ENTITY_LABELS,
    threshold=GLINER_THRESHOLD,
    include_confidence=GLINER_INCLUDE_CONFIDENCE,
)

print("GLiNER docs predicted:", len(gliner_predictions))
print("GLiNER total entities:", sum(len(v["entities"]) for v in gliner_predictions.values()))


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Loaded GLiNER adapter: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\ner\models\gliner_v2_finetuned\best


GLiNER2 inference (th=0.2): 100%|██████████| 40/40 [00:24<00:00,  1.61it/s]

GLiNER docs predicted: 40
GLiNER total entities: 1552


## 6) Ensemble span-level
Regole (semplici ma efficaci):
- Parto da tutte le entità BERT
- Aggiungo entità GLiNER **non-overlap** con BERT (stessa `location`) e con `score >= GLINER_MIN_ADD_SCORE`
- Se overlap e *stessa label* e GLiNER **contiene** lo span BERT, posso espandere (opzionale)
- Alla fine faccio un pruning overlap: **BERT ha priorità**

In [13]:
## Ensemble merge: start from BERT, add/expand with GLiNER, final prune (BERT priority)
from collections import defaultdict


def overlaps_1char(a: Dict[str, Any], b: Dict[str, Any]) -> bool:
    return not (int(a["end_idx"]) < int(b["start_idx"]) or int(b["end_idx"]) < int(a["start_idx"]))


def contains_span(outer: Dict[str, Any], inner: Dict[str, Any]) -> bool:
    return int(outer["start_idx"]) <= int(inner["start_idx"]) and int(outer["end_idx"]) >= int(inner["end_idx"])


def _with_source(e: Dict[str, Any], source: str) -> Dict[str, Any]:
    out = dict(e)
    out["_source"] = source
    out["_score"] = float(e.get("score", 1.0)) if "score" in e else 1.0
    return out


def final_prune_keep_best(ents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Keep a non-overlapping set per location.
    Priority: BERT > GLINER, then longer span, then higher score.
    """

    def sort_key(e):
        src_pri = 2 if e.get("_source") == "bert" else 1
        length = int(e["end_idx"]) - int(e["start_idx"])
        score = float(e.get("_score", 1.0))
        return (-src_pri, -length, -score, int(e["start_idx"]), int(e["end_idx"]))

    ents = sorted(ents, key=sort_key)

    kept: List[Dict[str, Any]] = []
    for e in ents:
        if any((k["location"] == e["location"]) and overlaps_1char(k, e) for k in kept):
            continue
        kept.append(e)
    return kept


def ensemble_merge(
        bert_preds: Dict[str, Any],
        gliner_preds: Dict[str, Any],
        dataset: Dict[str, Any],
) -> Dict[str, Any]:
    out: Dict[str, Any] = {}

    for pmid, article in dataset.items():
        title = (article.get("metadata") or {}).get("title", "") or ""
        abstract = (article.get("metadata") or {}).get("abstract", "") or ""

        b_ents = bert_preds.get(pmid, {}).get("entities", [])
        g_ents = gliner_preds.get(pmid, {}).get("entities", [])

        # index BERT by location
        b_by_loc = defaultdict(list)
        for b in b_ents:
            b_by_loc[str(b["location"])].append(b)

        merged: List[Dict[str, Any]] = [_with_source(b, "bert") for b in b_ents]

        for g in g_ents:
            g_score = float(g.get("score", 1.0))
            if g_score < GLINER_MIN_ADD_SCORE:
                continue

            loc = str(g["location"])
            conflicts = [b for b in b_by_loc.get(loc, []) if overlaps_1char(b, g)]

            # if no overlap -> add
            if not conflicts:
                merged.append(_with_source(g, "gliner"))
                continue

            # overlap exists: optionally expand if same label and GLiNER contains BERT
            if ALLOW_EXPAND_SAME_LABEL and g_score >= EXPAND_MIN_SCORE:
                for b in conflicts:
                    if b["label"] == g["label"] and contains_span(g, b):
                        # replace that BERT span in merged (first match only)
                        new_merged = []
                        replaced = False
                        for e in merged:
                            if (
                                    (not replaced)
                                    and e.get("_source") == "bert"
                                    and e["location"] == b["location"]
                                    and e["label"] == b["label"]
                                    and int(e["start_idx"]) == int(b["start_idx"])
                                    and int(e["end_idx"]) == int(b["end_idx"])
                            ):
                                new_merged.append(_with_source(g, "gliner"))
                                replaced = True
                            else:
                                new_merged.append(e)
                        merged = new_merged
                        break
            # else: keep BERT, skip GLiNER

        if APPLY_FINAL_PRUNE:
            merged = final_prune_keep_best(merged)

        # build final schema (no internal fields); recompute text_span from offsets
        cleaned: List[Dict[str, Any]] = []
        for e in merged:
            loc = str(e["location"])
            text = title if loc == "title" else abstract
            s, t = int(e["start_idx"]), int(e["end_idx"])

            if text and 0 <= s <= t < len(text):
                span = text[s: t + 1]
            else:
                span = e.get("text_span", "")

            cleaned.append(
                {
                    "start_idx": s,
                    "end_idx": t,
                    "location": loc,
                    "text_span": span,
                    "label": str(e["label"]),
                }
            )

        out[pmid] = {"entities": cleaned}

    return out


## Run ensemble
ensemble_predictions = ensemble_merge(bert_predictions, gliner_predictions, dev_data)
print("Ensemble docs:", len(ensemble_predictions))
print("Ensemble total entities:", sum(len(v["entities"]) for v in ensemble_predictions.values()))


Ensemble docs: 40
Ensemble total entities: 1555


##  Save predictions

In [14]:
out_path = OUT_PRED_DIR / "ensemble_ner.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(ensemble_predictions, f, ensure_ascii=False, indent=2)
print("Saved:", out_path)

Saved: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\predictions\ensemble_ner.json


##  EVALUATION

In [15]:
import copy
import json
from pathlib import Path
from typing import Any, Dict, Tuple

# =========================
# OFFICIAL EVALUATOR (COPIED)
# =========================

LEGAL_ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
]

def remove_duplicated_entities(predictions: dict) -> None:
    removed_count = 0
    for pmid in list(predictions.keys()):
        seen = set()
        deduped = []
        for ent in predictions[pmid]["entities"]:
            # OFFICIAL: key WITHOUT label
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        predictions[pmid]["entities"] = deduped

    if removed_count > 0:
        print(f"=== Removed {removed_count} duplicated entities from predictions ===")

def remove_overlapping_entities(predictions: dict) -> None:
    removed_count = 0

    for pmid in list(predictions.keys()):
        original_len = len(predictions[pmid]["entities"])

        groups = {"title": [], "abstract": []}
        for ent in predictions[pmid]["entities"]:
            loc = ent["location"]
            groups[loc].append(ent)

        keepers = set()
        for loc in groups:
            group = sorted(groups[loc], key=lambda e: e["start_idx"])

            clusters = []
            cluster = []
            current_end = None

            for ent in group:
                if not cluster:
                    cluster = [ent]
                    current_end = ent["end_idx"]
                else:
                    # OFFICIAL: overlap if ent.start < current_end
                    if ent["start_idx"] < current_end:
                        cluster.append(ent)
                        if ent["end_idx"] > current_end:
                            current_end = ent["end_idx"]
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = ent["end_idx"]

            if cluster:
                clusters.append(cluster)

            for clust in clusters:
                longest = clust[0]
                max_len = longest["end_idx"] - longest["start_idx"]
                for ent in clust[1:]:
                    length = ent["end_idx"] - ent["start_idx"]
                    if length > max_len:
                        longest = ent
                        max_len = length

                keepers.add((longest["start_idx"], longest["end_idx"], longest["location"]))

        deduped = []
        for ent in predictions[pmid]["entities"]:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key in keepers:
                deduped.append(ent)
                keepers.remove(key)

        predictions[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))

    if removed_count > 0:
        print(f"=== Removed {removed_count} overlapping entities ===")

def eval_submission_6_1_NER_from_predictions(
    predictions: Dict[str, Any],
    ground_truth: Dict[str, Any],
) -> Tuple[float, float, float, float, float, float]:
    """
    Same as official eval_submission_6_1_NER(path),
    but takes predictions dict directly (no file IO).
    """
    # IMPORTANT: the official functions mutate predictions -> deep copy
    predictions = copy.deepcopy(predictions)

    remove_duplicated_entities(predictions)
    remove_overlapping_entities(predictions)

    ground_truth_NER = {}
    count_annotated_entities_per_label = {}

    for pmid, article in ground_truth.items():
        ground_truth_NER.setdefault(pmid, [])
        for entity in article["entities"]:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"])

            entry = (start_idx, end_idx, location, text_span, label)
            ground_truth_NER[pmid].append(entry)

            count_annotated_entities_per_label[label] = count_annotated_entities_per_label.get(label, 0) + 1

    count_predicted_entities_per_label = {lab: 0 for lab in count_annotated_entities_per_label.keys()}
    count_true_positives_per_label = {lab: 0 for lab in count_annotated_entities_per_label.keys()}

    for pmid in predictions.keys():
        entities = predictions[pmid]["entities"]
        for entity in entities:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"])

            if label not in LEGAL_ENTITY_LABELS:
                raise NameError(f'{pmid} - Illegal label {label} for entity: {entity}')

            if label in count_predicted_entities_per_label:
                count_predicted_entities_per_label[label] += 1

            entry = (start_idx, end_idx, location, text_span, label)
            if entry in ground_truth_NER[pmid]:
                count_true_positives_per_label[label] += 1

    count_annotated_entities = sum(count_annotated_entities_per_label.values())
    count_predicted_entities = sum(count_predicted_entities_per_label.values())
    count_true_positives = sum(count_true_positives_per_label.values())

    micro_precision = count_true_positives / (count_predicted_entities + 1e-10)
    micro_recall = count_true_positives / (count_annotated_entities + 1e-10)
    micro_f1 = 2 * ((micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10))

    precision = recall = f1 = 0.0
    n = 0
    for label in count_annotated_entities_per_label.keys():
        n += 1
        current_precision = count_true_positives_per_label[label] / (count_predicted_entities_per_label[label] + 1e-10)
        current_recall = count_true_positives_per_label[label] / (count_annotated_entities_per_label[label] + 1e-10)
        precision += current_precision
        recall += current_recall
        f1 += 2 * ((current_precision * current_recall) / (current_precision + current_recall + 1e-10))

    precision /= n
    recall /= n
    f1 /= n

    return precision, recall, f1, micro_precision, micro_recall, micro_f1


def official_metrics_dict(
    predictions: Dict[str, Any],
    ground_truth: Dict[str, Any],
) -> Dict[str, float]:
    p, r, f1, mp, mr, mf1 = eval_submission_6_1_NER_from_predictions(predictions, ground_truth)
    return {
        "macro_precision": float(p),
        "macro_recall": float(r),
        "macro_f1": float(f1),
        "micro_precision": float(mp),
        "micro_recall": float(mr),
        "micro_f1": float(mf1),
    }


In [16]:
print("BERT-only:", official_metrics_dict(bert_predictions, dev_data))
print("GLiNER-only:", official_metrics_dict(gliner_predictions, dev_data))
print("ENSEMBLE:", official_metrics_dict(ensemble_predictions, dev_data))


BERT-only: {'macro_precision': 0.8678949614516107, 'macro_recall': 0.7136848455066089, 'macro_f1': 0.7676700326451387, 'micro_precision': 0.8867346938774605, 'micro_recall': 0.7779767233660897, 'micro_f1': 0.8288030519291519}
=== Removed 235 duplicated entities from predictions ===
=== Removed 68 overlapping entities ===
GLiNER-only: {'macro_precision': 0.39414148760613793, 'macro_recall': 0.4991698399652914, 'macro_f1': 0.40316490980901337, 'micro_precision': 0.46597277822254074, 'micro_recall': 0.5210384959713051, 'micro_f1': 0.4919695688427598}
ENSEMBLE: {'macro_precision': 0.5284353708789211, 'macro_recall': 0.8246200360886219, 'macro_f1': 0.6203278589747598, 'micro_precision': 0.616720257234687, 'micro_recall': 0.8585496866606214, 'micro_f1': 0.7178143712087748}


#### threshold gliner

In [17]:
def run_ensemble_with_gliner_threshold_official(th: float) -> Dict[str, Any]:
    gl_preds = predict_dataset_gliner2(
        extractor=gliner_model,
        dataset=dev_data,
        labels=ENTITY_LABELS,
        threshold=th,
    )
    ens = ensemble_merge(bert_predictions, gl_preds, dev_data)
    m = official_metrics_dict(ens, dev_data)
    return {"th": th, **m}

ths = [0.15, 0.18, 0.20, 0.22, 0.25, 0.30, 0.33]
results = [run_ensemble_with_gliner_threshold_official(th) for th in ths]
results


GLiNER2 inference (th=0.33): 100%|██████████| 40/40 [00:31<00:00,  1.28it/s]


[{'th': 0.15,
  'macro_precision': 0.5284353708789211,
  'macro_recall': 0.8246200360886219,
  'macro_f1': 0.6203278589747598,
  'micro_precision': 0.616720257234687,
  'micro_recall': 0.8585496866606214,
  'micro_f1': 0.7178143712087748},
 {'th': 0.18,
  'macro_precision': 0.5284353708789211,
  'macro_recall': 0.8246200360886219,
  'macro_f1': 0.6203278589747598,
  'micro_precision': 0.616720257234687,
  'micro_recall': 0.8585496866606214,
  'micro_f1': 0.7178143712087748},
 {'th': 0.2,
  'macro_precision': 0.5284353708789211,
  'macro_recall': 0.8246200360886219,
  'macro_f1': 0.6203278589747598,
  'micro_precision': 0.616720257234687,
  'micro_recall': 0.8585496866606214,
  'micro_f1': 0.7178143712087748},
 {'th': 0.22,
  'macro_precision': 0.53511741305597,
  'macro_recall': 0.8246200360886219,
  'macro_f1': 0.6253869731329337,
  'micro_precision': 0.6259791122714996,
  'micro_recall': 0.8585496866606214,
  'micro_f1': 0.7240468100681977},
 {'th': 0.25,
  'macro_precision': 0.54280